## **Harnessing Machine Learning for Seismic Phase Picking in the cloud**

Seismology is exploding with data. With thousands of sensors collecting terabytes of waveforms through networks like the SAGE facility, there’s more data than any human (or team of humans) can comb through by hand. That’s where machine learning (ML) comes in.

ML lets computers learn patterns from data, such as recognizing earthquake signals or picking out P and S wave arrivals, without being explicitly told the rules. Instead of writing rigid algorithms, we let the computer figure it out from examples.

Think of it as training a dog to sit. You don’t teach it every muscle to move; you reward it when it gets it right. Over time, it just learns. That’s exactly what we do when we train ML models on seismic data.

##### 🔹 **Train a model?**

“Training a model” means feeding it lots of labeled examples, like seismograms with known phase arrivals, and letting it learn the connection between the raw waveform and the arrival times.

At first, the model guesses randomly. But with each guess, it gets feedback (how wrong it was), and it adjusts its inner logic (called parameters) to do better next time. After enough rounds, the model gets pretty good at spotting patterns — even in brand new data it's never seen before.

##### 🔹 **Meet U-net**

When it comes to finding where something is inside messy data (like an earthquake hiding in noise), one of the most effective tools is called the U-Net. Originally created for segmenting medical images (like finding tumors in MRI scans), U-Net has found a new job in seismology: picking out seismic phases from waveform data.

How does it work? Think of it like this:

1. 👀 It zooms out to understand the big picture (what's happening over time).
2. 🔍 Then it zooms back in, focusing on fine details to pinpoint exactly where the interesting stuff is — like a P-wave arrival.

Even better, it brings along notes from the zoomed-out view so it doesn’t forget anything. This “zoom out and back in” strategy is why it's shaped like a U — wide on the sides, narrow in the middle.

U-Net works well on seismograms for a few key reasons:

- It combines context and detail — it sees the whole trace and also zooms in on precise moments.
- It’s efficient — it can learn a lot even from a relatively small number of labeled waveforms.
- It’s precise — it highlights the exact sample or time a seismic phase arrives.
- It’s flexible — it works for 1D traces, 2D waveform images, or even full event catalogs.
- All this makes it a favorite for seismic phase picking and event detection.

Training a deep learning model like U-Net takes time, power, and lots of data. Traditionally, that meant setting up local machines, transferring files, and managing storage — all of which can slow you down.

But now, SAGE data is hosted in the cloud, and that changes everything.

- You can stream waveform data directly into your ML pipeline.
- Use cloud GPUs or TPUs to train models in parallel.
- Automate your workflows using tools like Dask, Kubernetes, or prefect.io.
- Reproduce and share everything with portable, containerized environments.

In short: cloud-optimized workflows make training ML models on seismic data faster, cheaper, and more scalable than ever before.

This is how a U-net architecture works

<img src="https://oup.silverchair-cdn.com/oup/backfile/Content_public/Journal/gji/216/1/10.1093_gji_ggy423/1/m_ggy423fig5.jpeg?Expires=1754919728&Signature=41lJnaVjzCCW1i7utY-eOebfBdjeh7ykYBMBIqT0yyFLj3PCn60KlQD96BDL8kuQRxugnvm3BUxljgkI6VB6UcMNjIFXKamdpVuDpzjgwjMNwiZCdjOr1ygjxpixGkMHfzFV1w0y9JkY60gwVSZvhe4BlGfjAuuTZb9-6gVcxpLvQtFiOddFB6QpOblUD4RVFezUgbeCTa2VA4OI6O91G7DrWGR7EtSP~QAkYhqG4l~VmQTLxcZWijRHDEr~XGcPXlgR77x7NtRschgq6IHf0i7lv8F4MfbkMnvIDmkAm5m1pUQT8Q2SSyji9hT~p-NAyKhSEXuGFodua-SCpXWytg__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA" alt="DASKscaling" width="800"/>


In [2]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torchinfo import summary

## Build the U-net

First, we will go down with the Encoder path and build the required fuction. What is the first step that you see in there? It is the convolution and Relu process but twice. We are going to call it double convolution layer. So first let's build this layer below:

In [ ]:
# Import the necessary module from PyTorch
import torch.nn as nn

# Define a class to represent a double convolution block
class ConvBlock(nn.Module): # inherit the base class for neural networks for PyTorch
    """
    ┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
    │ ConvBlock: The basic building brick                                                              │
    │ ------------------------------------------------------------------------------------------------ │
    │ • Two 1-D convolutions (doubleConv), each followed by ReLU.                                      │
    │ • Keeps the time length unchanged (padding=1) but learns richer representations (more channels). |                                    │
    │ • Re-using the same pattern everywhere keeps the code short and consistent.                      |                                    │
    └──────────────────────────────────────────────────────────────────────────────────────────────────┘
    """
    def __init__(self, 
                 in_channels,       # No of input channels (e.g., 1 for grayscale, 3 for RGB)
                 out_channels,      # No of output channels (i.e., number of feature maps).
                 kernel_size = 3,   # Size of the convolutional filter (default is 3x3).
                 padding = 1        # Padding added to both sides of input (default is 1 to preserve size).
                 ):
        super().__init__()  # Initialize the parent nn.Module class
        
        # ① First 1-D convolution.
        #    – Looks at a sliding 3-sample window (for kernel_size=3).
        #    – padding=1 so output length == input length.
        #    – Learns out_channel different “filters” (patterns) in parallel.
        conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)

        # ② ReLU: keeps only positive values → adds non-linearity.
        relu1 = nn.ReLU(inplace=True)

        # ③ repeat these two steps once more, so we have a 'two-layer feature extractor'
        conv2 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        relu2 = nn.ReLU(inplace=True)

        # Put the four layers into one tidy “Sequential” container.
        self.doubleConv = nn.Sequential(conv1, relu1, conv2, relu2)
        
    def forward(self, x):
        # Simply run x through the mini-network with four layers (Conv-ReLu+Conv-ReLu) defined above.
        return self.doubleConv(x) # Pass input through the doubleConv block

If you’re just getting started with CNNs or U-net, building a modular layer like this helps break down a complex architecture into simpler, reusable parts. Once you understand how a single block works, it's easier to scale this up to the full model with encoder and decoder layers.

In [ ]:
# now, we are going to create the entire U-net structure

class UNet1D(nn.Module): # inherit the base class for neural networks for PyTorch
    """
    ┌────────────────────────────────────────────────────────────────────────┐
    │ 1-D U-Net (originally designed for images, adapted to time series)     │
    │                                                                        │
    │ • Goal: predict a label for every time sample (e.g., “noise / P / S”). │
    │ • Shape convention: (batch, channels, length) — PyTorch’s default.     │
    │ • Two big parts:                                                       │
    │     Encoder (Down path): “What is present?”                            │
    │     Decoder (Up path)  : “Where exactly is it?”                        │
    │   Skip connections copy high-resolution info from encoder to decoder.  │
    └────────────────────────────────────────────────────────────────────────┘
    """
    def __init__(self, 
                 in_channels = 3,               # e.g., 3-component seismogram
                 out_channels = 3,              # e.g., P-wave, S-wave, noise
                 features = [16, 32, 64, 182]   # network width; doubles every step by default
                 ):
        super().__init__()

        # ==============================
        # 1️⃣ Downsampling Path (ENCODER)
        # ==============================
        # Build the ENCODER (“Downs”) — series of ConvBlock + MaxPool  
        #  • Each ConvBlock learns richer features                     
        #  • MaxPool (done later in `forward`) halves time resolution
        #      doubling the “receptive field” (context window).
        # ---------------------------------------------------------------
        self.downs = nn.ModuleList()
        for feat in features:
            self.downs.append(ConvBlock(in_channels, feat)) # using the ConvBlock function we defined earlier
            in_channels = feat          # update in_channels for the next block where we are incrasing the features

        # ============================================
        # 2️⃣ Bottleneck (connects ENCODER & DECODER)
        # ============================================
        # bottleneck refers to the deepest layer in the U-Net, connects encoder and decoder
        #    • Sees the shortest signal (most compressed) but richest channels.
        #    • Doubles channels one last time.
        # --------------------------------------------------------------------------------------
        self.bottleneck = ConvBlock(features[-1], features[-1]*2)

        # ==============================
        # 3️⃣ Upsampling path (DECODER)
        # ==============================
        # Build the DECODER (“Ups”) — mirror of encoder
        #    For every level we create two layers:
        #       a) ConvTranspose1d for learnable upsampling (×2 length)
        #       b) ConvBlock to fuse the upsampled data with a skip connection
        # ------------------------------------------------------------------------
        self.ups   = nn.ModuleList()
        for feat in reversed(features):         # traverse 128→64→32→16
            # a) Up-convolution (upsample via transposed convolution): halves channels, doubles length
            self.ups.append(nn.ConvTranspose1d(feat*2, feat, kernel_size = 2, stride = 2))
            # b) ConvBlock: input has 2×feat channels (feat from up + feat skip)            
            self.ups.append(ConvBlock(feat*2, feat))

        # ===========================
        # Final output convolution
        # ===========================
        #    • Acts like a fully-connected layer applied at each time step.
        # ----------------------------------------------------------------------
        self.final_conv = nn.Conv1d(features[0], out_channels, kernel_size=1)

    # ════════════════════════════════════════════════════════════════════════
    # Forward pass: Encoder ➜ Bottleneck ➜ Decoder ➜ Classifier
    # ════════════════════════════════════════════════════════════════════════
    def forward(self, x):
        skip_stack = []         # will collect encoder outputs for skip connections

        # ---------------- Encoder ----------------
        for down in self.downs:
            x = down(x)                             # ConvBlock (keeps length)
            skip_stack.append(x)                    # save high-res features
            x = F.maxpool1d(x, kernel_size = 2)     # ↓2: halve length, double context
        
        # --------------- Bottleneck ---------------
        x = self.bottleneck(x)

        # Reverse list so the first pop corresponds to the last encoder block
        skip_stack = skip_stack[::-1]

        # ---------------- Decoder ----------------
        # Iterate pair-wise: (upconv, convblock), (upconv, convblock), ...
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)                # (a) learnable upsampling
            skip_conn = skip_stack[idx//2]      # (b) matching skip feature map

            # If the lengths differ by 1 (can happen with odd numbers),
            # right-pad the upsampled tensor so they match.
            if x.shape[-1] != skip_conn.shape[-1]:
                x = F.pad(x, (0, skip_conn.shape[-1] - x.shape[-1]))

            # (c) Concatenate along channel dimension: [skip | upsampled]            
            x = torch.cat((skip_conn, x), dim = 1)
            x = self.ups[idx+1](x)
        
        x = self.final_conv(x)          # raw scores per class
        return F.softmax(x, dim = 1)    # convert to probabilities

SyntaxError: invalid syntax (2177356388.py, line 87)